# Housing Price Prediction - Model Training Notebook

This notebook contains the exploratory data analysis and model training pipeline for the housing price prediction API.

## Steps:
1. Load and explore the dataset
2. Feature engineering
3. Train Linear Regression model
4. Evaluate and save the model


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import pickle
from datetime import datetime, timezone

## 1. Load and Explore Dataset

In [ ]:
DATA_PATH = Path('../data/housing.csv')
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe()

## 2. Feature Engineering

Exclude `id` and `price` from features. `price` is the target variable.

In [ ]:
FEATURE_COLUMNS = [
    'square_footage', 'bedrooms', 'bathrooms', 'year_built',
    'lot_size', 'distance_to_city_center', 'school_rating'
]
TARGET_COLUMN = 'price'

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

print(f'Features: {FEATURE_COLUMNS}')
print(f'Target: {TARGET_COLUMN}')

## 3. Train Linear Regression Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)
print('Model trained successfully!')

## 4. Evaluate Model

In [ ]:
y_pred = model.predict(X_test_scaled)

r2 = round(r2_score(y_test, y_pred), 4)
rmse = round(float(np.sqrt(mean_squared_error(y_test, y_pred))), 2)
mae = round(float(mean_absolute_error(y_test, y_pred)), 2)

print(f'R-squared: {r2}')
print(f'RMSE: {rmse}')
print(f'MAE: {mae}')

## 5. Save Model

In [ ]:
MODEL_PATH = Path('../model/model.pkl')
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

coefficients = {
    col: round(float(coef), 4)
    for col, coef in zip(FEATURE_COLUMNS, model.coef_)
}
intercept = round(float(model.intercept_), 4)

data = {
    'model': model,
    'scaler': scaler,
    'coefficients': coefficients,
    'intercept': intercept,
    'metrics': {'r_squared': r2, 'rmse': rmse, 'mae': mae},
    'training_date': datetime.now(timezone.utc).isoformat(),
    'n_samples': len(df),
}

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(data, f)

print(f'Model saved to {MODEL_PATH}')